In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Note: Ensure you run 'pip install imbalanced-learn' in your local environment for SMOTE
from imblearn.over_sampling import SMOTE 

# =====================================================================
# TASK 1: Load and Inspect Dataset
# =====================================================================
print("--- TASK 1: INITIAL DATA INSPECTION ---")
# Load the generated Zomato dataset
df = pd.read_csv('zomato_restaurants.csv')

print("\nFirst 5 Rows:")
print(df.head())

print("\nDataset Information (.info()):")
df.info()

print("\nSummary Statistics (.describe()):")
print(df.describe(include='all'))
print("\n" + "="*60 + "\n")


# =====================================================================
# TASK 2: Missing Values and Outliers Handling
# =====================================================================
print("--- TASK 2: CLEANING MISSING VALUES & OUTLIERS ---")
df_cleaned = df.copy()

# 1. Impute Missing Values using Column Medians
df_cleaned['cost'] = df_cleaned['cost'].fillna(df_cleaned['cost'].median())
df_cleaned['rating'] = df_cleaned['rating'].fillna(df_cleaned['rating'].median())

# 2. Outlier Detection using the Interquartile Range (IQR) Method for 'cost'
Q1 = df_cleaned['cost'].quantile(0.25)
Q3 = df_cleaned['cost'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_before = ((df_cleaned['cost'] < lower_bound) | (df_cleaned['cost'] > upper_bound)).sum()
print(f"Identified {outliers_before} outlier records in 'cost' column.")

# Cap/Clip the extreme values to the IQR bounds
df_cleaned['cost'] = np.clip(df_cleaned['cost'], lower_bound, upper_bound)
print("Outliers capped successfully using IQR boundaries.")
print("\n" + "="*60 + "\n")


# =====================================================================
# TASK 3: Categorical Encoding & Feature Scaling
# =====================================================================
print("--- TASK 3: ENCODING AND SCALING CORRIDORS ---")
df_encoded = df_cleaned.copy()

# 1. Encode Categorical Columns using LabelEncoder
le_cuisine = LabelEncoder()
df_encoded['cuisine'] = le_cuisine.fit_transform(df_encoded['cuisine'])

le_location = LabelEncoder()
df_encoded['location'] = le_location.fit_transform(df_encoded['location'])

# 2. Scale Numeric Columns using StandardScaler
scaler = StandardScaler()
df_encoded[['cost', 'rating']] = scaler.fit_transform(df_encoded[['cost', 'rating']])

print("Categorical variables encoded and numerical attributes standardized.")
print("\n" + "="*60 + "\n")


# =====================================================================
# TASK 4: Train-Test Split and Model Benchmarking
# =====================================================================
print("--- TASK 4: MODEL TRAINING & ROC-AUC COMPARISON ---")

# Define the Target Variable: 1 if rating > 4.0, else 0
# IMPORTANT DATA SCIENCE NOTE: Since 'rating' was used to construct our target variable, 
# keeping 'rating' inside our features matrix (X) creates perfect TARGET LEAKAGE. 
# To build a realistic model, we drop 'rating' from the feature matrix X.
y = (df_cleaned['rating'] > 4.0).astype(int)
X = df_encoded[['cost', 'cuisine', 'location']]

print(f"Target Class Distribution:\n{y.value_counts()}")

# Split into 80% Training and 20% Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1. Train Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)
lr_auc = roc_auc_score(y_test, lr_model.predict_proba(X_test)[:, 1])

# 2. Train Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])

print(f"\nLogistic Regression Test ROC-AUC : {lr_auc:.4f}")
print(f"Random Forest Test ROC-AUC       : {rf_auc:.4f}")

better_model = "Logistic Regression" if lr_auc > rf_auc else "Random Forest"
print(f"🥇 Winner: {better_model} performed better on the test set.")
print("\n" + "="*60 + "\n")


# =====================================================================
# TASK 5: Imbalance Management (SMOTE) & Hyperparameter Optimization
# =====================================================================
print("--- TASK 5: SMOTE OVERSAMPLING & GRID SEARCH ---")

# 1. Address Class Imbalance via SMOTE on the Training set
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"Resampled Training Target Distribution:\n{y_train_res.value_counts()}")

# 2. Set up Hyperparameter Tuning Space for Random Forest
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7, None]
}

# 3. Apply GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_train_res, y_train_res)

# 4. Evaluate Optimized Model
best_rf = grid_search.best_estimator_
tuned_rf_auc = roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1])

print(f"\nOptimized Parameters found via GridSearch: {grid_search.best_params_}")
print(f"Best CV Cross-Validation ROC-AUC Score   : {grid_search.best_score_:.4f}")
print(f"Final Tuned Random Forest Test ROC-AUC   : {tuned_rf_auc:.4f}")